In [0]:
%pip install databricks-vectorsearch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 12.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.3
    Not uninstalling requests at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-22796379-bd75-4d79-b6ea-70ee5201c255
    Can't uninstall 'requests'. No files were found to uninstall.
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.4
    Not uninstalling protobuf at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-22796379-bd75-4d79-b6ea-70ee5201c255
    Can't uninstall 'protobuf'. No files were found to uninstall.
  Attempting uninstall: mlflow-skinny
    Found existing installation: mlflow-skinny 3.8.1
    Not uninstalling mlflow-skinny at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-22796379-bd75-4d79-

In [0]:
%restart_python

In [0]:
from databricks.vector_search.client import VectorSearchClient

client = VectorSearchClient()

for ep in client.list_endpoints().get("endpoints", []):
    print(ep["name"])

/home/spark-3f4ad0d0-e5a5-40e5-a2dc-2d/.ipykernel/16044/command-6004898166018496-1203741396:1: DeprecationWarning: databricks-vectorsearch is deprecated and has been renamed to databricks-ai-search. Imports under 'databricks.vector_search.*' will continue to work as a thin re-export of 'databricks.ai_search.*', but new code should switch to 'pip install databricks-ai-search' and 'from databricks.ai_search.* import ...'.
  from databricks.vector_search.client import VectorSearchClient


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
movielens
demo_vector_endpoint


In [0]:
catalog = "workspace"
schema = "default"

table_name = "movies_embedding" # actual table, we use automatic embedding by databricks 
index_name = "movies_index" # index is searchable vector
endpoint_name = "movielens" # one end point is used by many indexes

In [0]:
full_table_name = f"{catalog}.{schema}.{table_name}"
full_index_name = f"{catalog}.{schema}.{index_name}"

print(full_table_name)
print(full_index_name)

workspace.default.movies_embedding
workspace.default.movies_index


In [0]:
# create end point
# run this code onces, else you need to search end point, try to get the desired endpoint or create if that does not exists

from databricks.vector_search.client import VectorSearchClient

client = VectorSearchClient()

client.create_endpoint(
    name=endpoint_name,
    endpoint_type="STANDARD"
)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


{'name': 'movielens',
 'creator': 'ggknco@gmail.com',
 'creation_timestamp': 1780888548672,
 'last_updated_timestamp': 1780888548672,
 'endpoint_type': 'STANDARD',
 'last_updated_user': 'ggknco@gmail.com',
 'id': '5cdcea81-4a11-465e-b568-1a834595ccf6',
 'endpoint_status': {'state': 'ONLINE'},
 'num_indexes': 0}

In [0]:
# now list created end points
for ep in client.list_endpoints().get("endpoints", []):
    print(ep["name"])

movielens
demo_vector_endpoint


In [0]:
# wait for endpoint is ONLINE

import time

while True:
    endpoint = client.get_endpoint(endpoint_name)

    state = endpoint["endpoint_status"]["state"]

    print("Status:", state)

    if state == "ONLINE":
        break

    time.sleep(30)

Status: ONLINE


In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {full_table_name}
(
    movieId INT,
    title STRING,
    genres STRING
)
USING DELTA
TBLPROPERTIES (
    delta.enableChangeDataFeed = true
)
""")

DataFrame[]

In [0]:
spark.sql(f"""
INSERT INTO {full_table_name}
VALUES
(1,'Toy Story','Animation|Comedy'),
(2,'Avatar','Sci-Fi|Adventure'),
(3,'Interstellar','Sci-Fi|Space'),
(4,'Star Wars','Sci-Fi|Action')
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
display(
    spark.table(full_table_name)
)

movieId,title,genres
1,Toy Story,Animation|Comedy
2,Avatar,Sci-Fi|Adventure
3,Interstellar,Sci-Fi|Space
4,Star Wars,Sci-Fi|Action


In [0]:
# create delta sync index
# it is called sync index, because it is synced with delta table , any changes on delta table reflected on index when the trigger runs

# pipeline_type can be TRIGGERED or STREAMING
# TRIGGERED need explicit call to run the index to update 
# this need source table to be change data feed enabled
client.create_delta_sync_index(
    endpoint_name=endpoint_name,
    index_name=full_index_name,
    primary_key="movieId",
    source_table_name=full_table_name,
    pipeline_type="TRIGGERED",
    embedding_source_column="title",
    embedding_model_endpoint_name="databricks-gte-large-en"
)

"""
we could also have a table that has embedding vector as column, there we use embedding vector column, not a source column.
 embedding_vector_column="embedding",
    embedding_dimension=len(vectors[0])
"""

In [0]:
display(
    spark.table(full_table_name)
)

movieId,title,genres
1,Toy Story,Animation|Comedy
2,Avatar,Sci-Fi|Adventure
3,Interstellar,Sci-Fi|Space
4,Star Wars,Sci-Fi|Action


In [0]:
# table has data
display(
    spark.table(full_table_name)
)

movieId,title,genres
1,Toy Story,Animation|Comedy
2,Avatar,Sci-Fi|Adventure
3,Interstellar,Sci-Fi|Space
4,Star Wars,Sci-Fi|Action


In [0]:
import json
# index description , we are checking status
idx = client.get_index(
    endpoint_name=endpoint_name,
    index_name=full_index_name
)

desc = idx.describe()

print ("Index ready? ", desc.get("status", {}).get("ready"))

print(json.dumps(desc, indent=2))



Index ready?  True
{
  "name": "workspace.default.movies_index",
  "endpoint_name": "movielens",
  "primary_key": "movieId",
  "index_type": "DELTA_SYNC",
  "delta_sync_index_spec": {
    "source_table": "workspace.default.movies_embedding",
    "embedding_source_columns": [
      {
        "name": "title",
        "embedding_model_endpoint_name": "databricks-gte-large-en"
      }
    ],
    "pipeline_type": "TRIGGERED",
    "pipeline_id": "bc509283-1250-4e9f-a49c-bcf59eb478aa"
  },
  "status": {
    "detailed_state": "ONLINE_TRIGGERED_UPDATE",
    "message": "Index is online but is currently is in the process of re-syncing initial data. Check latest status: https://dbc-d2625d47-674e.cloud.databricks.com/explore/data/workspace/default/movies_index",
    "indexed_row_count": 4,
    "triggered_update_status": {
      "last_processed_commit_version": 3,
      "last_processed_commit_timestamp": "2026-06-08T04:03:02Z",
      "triggered_update_progress": {
        "latest_version_currently_p

In [0]:
# first query, note, we still not yet indexed, you will get error like index , index-xyxaaa... not exists
# ERROR EXPECTED
try:
    results = idx.similarity_search(
        query_text="toy",
        columns=["movieId","title","genres"],
        num_results=5
    )

    print(results)
except Exception as e:
    print(e)
 

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Index index-140f9d3c-1b11-4774-8235-ea612d89362e does not exist


In [0]:
# this will trigger index syching
# may fail if index is not provisioned, ie end point isnot yet ready
# may fail if index isnot created, however idx.sync will take care all behind the scene
idx.sync()
# now go back and run the status cell above to check status to READY to be true

{}

In [0]:
# now use index to search
# change query texts like toy, wars
results = idx.similarity_search(
    query_text="wars",
    columns=["movieId","title","genres"],
    num_results=5
)

print(results)

import json
print(json.dumps(results, indent=2))

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
{'manifest': {'column_count': 4, 'columns': [{'name': 'movieId'}, {'name': 'title'}, {'name': 'genres'}, {'name': 'score'}]}, 'result': {'row_count': 4, 'data_array': [[4.0, 'Star Wars', 'Sci-Fi|Action', 0.5743672972557249], [2.0, 'Avatar', 'Sci-Fi|Adventure', 0.5014212494843842], [1.0, 'Toy Story', 'Animation|Comedy', 0.4907657233314682], [3.0, 'Interstellar', 'Sci-Fi|Space', 0.48491485484988145]]}}
{
  "manifest": {
    "column_count": 4,
    "columns": [
      {
        "name": "movieId"
      },
      {
        "name": "title"
      },
      {
        "name": "genres"
      },
      {
        "name": "score"
      }
    ]
  },
  "result": {
    "row_count": 4,
    "data_array": [
      [
        4.0,
        "Star Wars",
        "Sci-Fi|Action",
        0.5743672972557249
   

In [0]:
endpoint = client.get_endpoint(endpoint_name)

print(endpoint)

{'name': 'movielens', 'creator': 'ggknco@gmail.com', 'creation_timestamp': 1780888548672, 'last_updated_timestamp': 1780888548672, 'endpoint_type': 'STANDARD', 'last_updated_user': 'ggknco@gmail.com', 'id': '5cdcea81-4a11-465e-b568-1a834595ccf6', 'endpoint_status': {'state': 'ONLINE'}, 'num_indexes': 1, 'throughput_info': {'requested_concurrency': 2.0, 'current_concurrency': 2.0, 'current_concurrency_utilization_percentage': 2.0, 'change_request_state': 'CHANGE_SUCCESS', 'requested_num_replicas': 1, 'current_num_replicas': 1}}


In [0]:
spark.sql(f"""
INSERT INTO {full_table_name}
VALUES
(
    5,
    'The Martian',
    'Sci-Fi|Adventure'
)
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
display(
    spark.table(full_table_name)
)

movieId,title,genres
1,Toy Story,Animation|Comedy
2,Avatar,Sci-Fi|Adventure
3,Interstellar,Sci-Fi|Space
4,Star Wars,Sci-Fi|Action
5,The Martian,Sci-Fi|Adventure


In [0]:
# we will not see martian, as it is not indexed, index was not trigged
results = idx.similarity_search(
    query_text="Martian",
    columns=["movieId","title","genres"],
    num_results=5
)

print(results)

import json
print(json.dumps(results, indent=2))

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
{'manifest': {'column_count': 4, 'columns': [{'name': 'movieId'}, {'name': 'title'}, {'name': 'genres'}, {'name': 'score'}]}, 'result': {'row_count': 4, 'data_array': [[3.0, 'Interstellar', 'Sci-Fi|Space', 0.6202237591461138], [2.0, 'Avatar', 'Sci-Fi|Adventure', 0.586431689143632], [4.0, 'Star Wars', 'Sci-Fi|Action', 0.5762023264735009], [1.0, 'Toy Story', 'Animation|Comedy', 0.5402878141437913]]}}
{
  "manifest": {
    "column_count": 4,
    "columns": [
      {
        "name": "movieId"
      },
      {
        "name": "title"
      },
      {
        "name": "genres"
      },
      {
        "name": "score"
      }
    ]
  },
  "result": {
    "row_count": 4,
    "data_array": [
      [
        3.0,
        "Interstellar",
        "Sci-Fi|Space",
        0.6202237591461138
   

In [0]:
client.sync_index(
    endpoint_name,
    full_index_name
)

In [0]:
idx.sync()

{}

In [0]:
results = idx.similarity_search(
    query_text="astronaut stranded on mars",
    columns=["movieId","title","genres"],
    num_results=5
)

import pprint
pprint.pprint(results)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
{'manifest': {'column_count': 4,
              'columns': [{'name': 'movieId'},
                          {'name': 'title'},
                          {'name': 'genres'},
                          {'name': 'score'}]},
 'result': {'data_array': [[5.0,
                            'The Martian',
                            'Sci-Fi|Adventure',
                            0.6840074948337858],
                           [3.0,
                            'Interstellar',
                            'Sci-Fi|Space',
                            0.5511513936695542],
                           [4.0,
                            'Star Wars',
                            'Sci-Fi|Action',
                            0.49330869364980984],
                           [2.0,
                           